# Motive Raw CSV QC — Layers 1–2

Interactive front-end for parser validation, gap/missingness QC, unlabeled burden, and frame-level QC masks.

**Workflow**
1. Setup and review config
2. Run Layer 1 (parse + marker inventory)
3. Layer 1 validation checkpoint
4. Run Layer 2 (gaps, masks, outputs)
5. Layer 2 validation checkpoint
6. Explore results (tabs below)
7. Sign off and write validation log

**In scope:** raw marker XYZ CSV QC, gap structure, unlabeled burden, `frame_qc_mask.csv`, markdown report.

**Not in scope (later layers):** artifact screening, BVH parsing/mapping, automated exclusions.

All computation lives in `motive_raw_qc.py`. This notebook orchestrates steps and displays outputs for researcher validation.

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import HTML, Image, Markdown, clear_output, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "motive_raw_qc.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import motive_raw_qc as mqc

versions = pd.DataFrame(
    {
        "package": ["motive_raw_qc", "pandas", "numpy", "xarray", "matplotlib", "ipywidgets"],
        "version": [
            mqc.__version__,
            pd.__version__,
            __import__("numpy").__version__,
            __import__("xarray").__version__,
            plt.matplotlib.__version__,
            widgets.__version__,
        ],
    }
)
display(versions)

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = mqc.load_config(CONFIG_PATH)
config["_base_dir"] = PROJECT_ROOT

display_config = {k: v for k, v in config.items() if not k.startswith("_")}
display(Markdown(f"### Resolved config\n\n```yaml\n{yaml.safe_dump(display_config, sort_keys=False)}```"))

tuning_rows = [
    {"section": "gaps.thresholds_seconds", "setting": k, "value": v}
    for k, v in config["gaps"]["thresholds_seconds"].items()
]
mask_cfg = config.get("frame_qc_mask", {})
for key, value in mask_cfg.items():
    tuning_rows.append({"section": "frame_qc_mask", "setting": key, "value": value})
tuning_rows.append(
    {
        "section": "markers",
        "setting": "include_unlabeled_markers",
        "value": config["markers"].get("include_unlabeled_markers"),
    }
)
tuning_rows.append(
    {
        "section": "markers",
        "setting": "unlabeled_name_patterns",
        "value": config["markers"].get("unlabeled_name_patterns", []),
    }
)
for plot_name, enabled in config["outputs"]["plots"].items():
    if plot_name != "enabled":
        tuning_rows.append({"section": "outputs.plots", "setting": plot_name, "value": enabled})

display(Markdown("### Key tuning settings (read-only)"))
display(pd.DataFrame(tuning_rows))
display(Markdown("*Edit `config.yaml`, then re-run from the Layer 1 or Layer 2 cells.*"))

In [ ]:
try:
    layer1 = mqc.run_layer1_parse(config, verbose=True)
except (
    mqc.ConfigValidationError,
    mqc.MotiveCSVParseError,
    mqc.SchemaValidationError,
    mqc.QCValidationError,
) as exc:
    display(pd.DataFrame([{"error_type": type(exc).__name__, "message": str(exc)}]))
    raise

session = layer1.session
md = session.metadata
display(
    Markdown(
        f"**Layer 1 status:** `{layer1.status}` | "
        f"frames: `{md['total_frames_observed']}` | "
        f"labeled: `{md['n_labeled_markers']}` | "
        f"unlabeled: `{md['n_unlabeled_markers']}` | "
        f"excluded non-marker cols: `{md.get('excluded_non_marker_column_count', 0)}` | "
        f"`raw_data_status`: `{md['raw_data_status']}`"
    )
)
display(layer1.tables["session_summary"])
print(f"Validation messages: {len(layer1.messages)}")

## Layer 1 validation checkpoint

Review before continuing to Layer 2:

- [ ] Observed frame count matches Motive export
- [ ] Effective frame rate is correct
- [ ] Marker count is plausible
- [ ] Labeled vs unlabeled separation is correct (including naming variants / subject prefix)
- [ ] Marker names match expected Motive labels
- [ ] Excluded non-marker column count is expected (rigid/skeleton/quaternion not parsed as markers)
- [ ] No rigid-body/skeleton/quaternion columns treated as raw markers
- [ ] `raw_data_status` is acceptable

**Decision:** approved / needs correction

In [ ]:
try:
    layer2 = mqc.run_layer2_gaps(session, config, verbose=True)
    files_written = mqc.write_outputs(layer1, layer2, config, verbose=True)
except mqc.QCValidationError as exc:
    display(pd.DataFrame([{"error_type": type(exc).__name__, "message": str(exc)}]))
    raise

OUTPUT_DIR = (PROJECT_ROOT / config["paths"]["output_dir"]).resolve()
REPORT_PATH = OUTPUT_DIR / "qc_report_summary.md"
summary = layer2.tables["session_summary"].iloc[0]
display(
    Markdown(
        f"**Layer 2 complete.** Wrote `{len(files_written)}` files to `{OUTPUT_DIR}`\n\n"
        f"Preprocessing status: `{summary['raw_qc_preprocessing_status']}` | "
        f"labeled missing: `{summary['missing_percent_labeled']}%` | "
        f"gaps >= 0.5 s (labeled): `{summary['n_gaps_ge_0p5s_labeled']}`"
    )
)

## Layer 2 validation checkpoint

Review before any Layer 3 work:

- [ ] `marker_quality_summary` marker count matches `marker_inventory`
- [ ] Gap start/end frames look correct on spot checks
- [ ] `duration_frames` is inclusive
- [ ] `duration_seconds = duration_frames / effective_frame_rate_hz`
- [ ] Gaps >= 0.2 s and >= 0.5 s match manual checks
- [ ] Labeled and unlabeled summaries are separated
- [ ] `unlabeled_marker_summary` values are plausible
- [ ] `frame_qc_mask` status counts make sense
- [ ] `gap_timeline.png` and `unlabeled_count_over_time.png` are readable
- [ ] `qc_report_summary.md` sections 1–3 populated; sections 4–5 correctly stubbed
- [ ] No gap filling, smoothing, or filtering was applied

**Decision:** approved / needs correction

In [ ]:
LARGE_GAP_S = config["gaps"]["thresholds_seconds"]["large_gap"]
MOD_GAP_S = config["gaps"]["thresholds_seconds"]["moderate_gap"]
CRITICAL_GROUPS = set(config.get("frame_quality", {}).get("critical_groups", []))
BIN_WIDTH_S = 30
OVERVIEW_LARGE_GAP_ROWS = 20

MAX_TABLE_ROWS = 500
GAP_EVENTS_VIEW_COLS = [
    "marker_name",
    "body_region_group",
    "gap_start_frame",
    "gap_end_frame",
    "duration_frames",
    "duration_seconds",
]


def _gap_events_df() -> pd.DataFrame:
    return layer2.tables.get("gap_events", pd.DataFrame())


def _labeled_gaps(df: pd.DataFrame, min_seconds: float) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    return df[df["is_labeled"] & (df["duration_seconds"] >= min_seconds)].copy()


def _gap_overview_large(df: pd.DataFrame) -> pd.DataFrame:
    gaps = _labeled_gaps(df, LARGE_GAP_S)
    if gaps.empty:
        return gaps
    cols = [
        "marker_name",
        "body_region_group",
        "duration_frames",
        "gap_start_frame",
        "gap_end_frame",
        "duration_seconds",
    ]
    return gaps[cols].sort_values("duration_seconds", ascending=False).head(
        OVERVIEW_LARGE_GAP_ROWS
    )


def _gap_timeline_bins(df: pd.DataFrame, duration_s: float) -> pd.DataFrame:
    gaps = _labeled_gaps(df, MOD_GAP_S)
    if duration_s <= 0:
        return pd.DataFrame(
            columns=[
                "window_start_s",
                "window_end_s",
                "n_gaps_ge_0p2s",
                "critical_gap_in_bin",
            ]
        )
    rows = []
    start = 0.0
    while start < duration_s:
        end = min(start + BIN_WIDTH_S, duration_s)
        in_bin = gaps[
            (gaps["gap_start_time_seconds"] >= start)
            & (gaps["gap_start_time_seconds"] < end)
        ]
        critical = False
        if not in_bin.empty and CRITICAL_GROUPS:
            critical = in_bin["body_region_group"].isin(CRITICAL_GROUPS).any()
        rows.append(
            {
                "window_start_s": round(start, 3),
                "window_end_s": round(end, 3),
                "n_gaps_ge_0p2s": int(len(in_bin)),
                "critical_gap_in_bin": bool(critical),
            }
        )
        start = end
    return pd.DataFrame(rows)


def _gap_events_researcher_view(df: pd.DataFrame) -> pd.DataFrame:
    gaps = _labeled_gaps(df, MOD_GAP_S)
    if gaps.empty:
        return gaps
    return gaps[GAP_EVENTS_VIEW_COLS].sort_values("gap_start_frame")


TABLE_OPTIONS = {
    "session_summary": lambda: layer2.tables["session_summary"],
    "marker_inventory": lambda: layer1.tables["marker_inventory"],
    "marker_quality_summary": lambda: layer2.tables["marker_quality_summary"],
    "gap_events_labeled_ge_0p2s": lambda: _gap_events_researcher_view(_gap_events_df()),
    "gap_summary_by_group": lambda: layer2.tables["gap_summary_by_group"],
    "unlabeled_marker_summary": lambda: layer2.tables["unlabeled_marker_summary"],
    "unlabeled_frame_counts": lambda: layer2.tables.get(
        "unlabeled_frame_counts", pd.DataFrame()
    ),
    "frame_qc_mask": lambda: layer2.tables["frame_qc_mask"],
}


def _mask_filter_df(df: pd.DataFrame, mask_filter: str) -> pd.DataFrame:
    if mask_filter == "all" or "qc_status" not in df.columns:
        return df
    return df[df["qc_status"] == mask_filter]


def render_overview():
    md = session.metadata
    summ = layer2.tables["session_summary"].iloc[0]
    unl = layer2.tables.get("unlabeled_marker_summary", pd.DataFrame())
    unl_row = unl.iloc[0] if not unl.empty else {}
    mask = layer2.tables.get("frame_qc_mask", pd.DataFrame())
    gap_df = _gap_events_df()
    mask_counts = (
        mask["qc_status"].value_counts().to_dict() if not mask.empty else {}
    )
    mask_lines = "<br>".join(f"{k}: {v}" for k, v in mask_counts.items()) or "(none)"
    qc_status = summ.get("raw_qc_preprocessing_status", "unknown")
    html = f"""
    <div style="font-family: sans-serif; line-height: 1.5;">
      <h3 style="margin-top:0;">Session</h3>
      <b>File:</b> {md['file_name']}<br>
      <b>Frames:</b> {md['total_frames_observed']} ({md['start_frame']}–{md['end_frame']}) @ {md['effective_frame_rate_hz']} Hz<br>
      <b>Markers:</b> {md['n_labeled_markers']} labeled, {md['n_unlabeled_markers']} unlabeled<br>
      <b>Raw data status:</b> {md['raw_data_status']}<br>
      <b>Preprocessing QC:</b> {qc_status}<br><br>
      <h3>Gaps (labeled)</h3>
      Missing: {summ['missing_percent_labeled']}% | gaps ≥0.2 s: {summ['n_gaps_ge_0p2s_labeled']} | ≥0.5 s: {summ['n_gaps_ge_0p5s_labeled']}<br>
      Union gap time (≥0.2 s): {summ.get('union_gap_seconds_ge_0p2_labeled', 'n/a')} s |
      Max critical-region gap: {summ.get('max_critical_gap_seconds_labeled', 'n/a')} s<br>
      Longest gap: {summ['longest_gap_seconds_labeled']} s ({summ['longest_gap_marker_labeled']})<br><br>
      <h3>Unlabeled burden</h3>
      Frames with unlabeled: {unl_row.get('frames_with_any_unlabeled', 0)} ({unl_row.get('percent_frames_with_any_unlabeled', 0)}%)<br>
      Longest burst: {unl_row.get('longest_unlabeled_burst_sec', 0)} s | overlap w/ labeled gaps: {unl_row.get('overlap_with_labeled_gaps', False)}<br><br>
      <h3>Frame QC mask</h3>
      {mask_lines}<br><br>
      <b>Output folder:</b> {OUTPUT_DIR}
    </div>
    """
    display(HTML(html))

    display(
        Markdown(
            """#### Preprocessing QC status (rule-based, not automatic exclusion)

- **acceptable** — Low labeled missingness; limited moderate+ gap time on the timeline; no severe critical-region gap.
- **caution** — Notable gap time or a critical-marker gap worth reviewing before Motive gap-fill/solve.
- **poor** — Substantial affected timeline (union moderate+ gap time at/above threshold) or severe critical-region gap.

Computed from `config.yaml` thresholds. Does not delete frames or block analysis."""
        )
    )
    display(
        Markdown(
            f"**Status reason:** {summ.get('raw_qc_status_reason', '')}  \n"
            f"**Metrics:** union_gap_time={summ.get('union_gap_seconds_ge_0p2_labeled', 'n/a')} s, "
            f"max_critical_gap={summ.get('max_critical_gap_seconds_labeled', 'n/a')} s, "
            f"n_gaps_ge_0p5={summ.get('n_gaps_ge_0p5s_labeled', 'n/a')}"
        )
    )

    display(Markdown(f"#### Large labeled gaps (>= {LARGE_GAP_S} s)"))
    large_tbl = _gap_overview_large(gap_df)
    if large_tbl.empty:
        display(Markdown(f"*No labeled gaps >= {LARGE_GAP_S} s.*"))
    else:
        display(large_tbl)

    duration_s = float(md.get("duration_seconds", 0))
    display(Markdown(f"#### Session time bins ({BIN_WIDTH_S} s) — labeled gaps >= {MOD_GAP_S} s"))
    bin_tbl = _gap_timeline_bins(gap_df, duration_s)
    display(bin_tbl)
    if not bin_tbl.empty and bin_tbl["n_gaps_ge_0p2s"].sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 3))
        centers = (bin_tbl["window_start_s"] + bin_tbl["window_end_s"]) / 2
        ax.bar(centers, bin_tbl["n_gaps_ge_0p2s"], width=BIN_WIDTH_S * 0.9, color="#4a5568")
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Gap count")
        ax.set_title("Labeled gaps >= moderate threshold per time bin")
        fig.tight_layout()
        display(fig)
        plt.close(fig)


def render_table(table_name: str, mask_filter: str = "all"):
    df = TABLE_OPTIONS[table_name]()
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        display(Markdown("*No data for this table.*"))
        return
    if table_name == "frame_qc_mask":
        df = _mask_filter_df(df, mask_filter)
    truncated = len(df) > MAX_TABLE_ROWS
    view = df.head(MAX_TABLE_ROWS) if truncated else df
    display(view)
    if truncated:
        display(Markdown(f"*Showing first {MAX_TABLE_ROWS} of {len(df)} rows.*"))


def render_plot(plot_name: str):
    figures = layer2.figures
    if plot_name not in figures:
        display(Markdown(f"*Plot not available: `{plot_name}`*"))
        return
    display(Image(filename=str(figures[plot_name])))


def render_report():
    if REPORT_PATH.exists():
        display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))
    else:
        display(Markdown("*Report not found. Run Layer 2 first.*"))


def render_messages():
    rows = [
        {"severity": m.severity, "code": m.code, "message": m.message}
        for m in layer1.messages + layer2.messages
    ]
    if rows:
        display(pd.DataFrame(rows))
    else:
        display(Markdown("*No validation messages.*"))


overview_out = widgets.Output()
tables_out = widgets.Output()
plots_out = widgets.Output()
report_out = widgets.Output()
messages_out = widgets.Output()

table_dd = widgets.Dropdown(
    options=list(TABLE_OPTIONS.keys()),
    value="session_summary",
    description="Table:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)
mask_dd = widgets.Dropdown(
    options=["all", "use", "caution", "exclude_or_review"],
    value="all",
    description="Mask filter:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)
plot_dd = widgets.Dropdown(
    options=sorted(layer2.figures.keys()) if layer2.figures else ["(none)"],
    description="Plot:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)

tables_panel = widgets.VBox([widgets.HBox([table_dd, mask_dd]), tables_out])
plots_panel = widgets.VBox([plot_dd, plots_out])


def _update_mask_visibility(*_):
    mask_dd.layout.display = None if table_dd.value == "frame_qc_mask" else "none"


def _on_table_change(change):
    _update_mask_visibility()
    with tables_out:
        clear_output(wait=True)
        render_table(table_dd.value, mask_dd.value)


def _on_mask_change(change):
    if table_dd.value == "frame_qc_mask":
        with tables_out:
            clear_output(wait=True)
            render_table(table_dd.value, mask_dd.value)


def _on_plot_change(change):
    with plots_out:
        clear_output(wait=True)
        if plot_dd.value != "(none)":
            render_plot(plot_dd.value)


table_dd.observe(_on_table_change, names="value")
mask_dd.observe(_on_mask_change, names="value")
plot_dd.observe(_on_plot_change, names="value")

with overview_out:
    render_overview()
with tables_out:
    render_table(table_dd.value, mask_dd.value)
with plots_out:
    if layer2.figures:
        render_plot(plot_dd.value)
with report_out:
    render_report()
with messages_out:
    render_messages()

_update_mask_visibility()

tabs = widgets.Tab(children=[overview_out, tables_panel, plots_panel, report_out, messages_out])
tabs.set_title(0, "Overview")
tabs.set_title(1, "Tables")
tabs.set_title(2, "Plots")
tabs.set_title(3, "Report")
tabs.set_title(4, "Messages")
display(Markdown("### Results explorer"))
display(tabs)

In [ ]:
validated_by_w = widgets.Text(
    description="Validated by:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
decision_w = widgets.Dropdown(
    options=["pending", "approved", "needs correction"],
    value="pending",
    description="Decision:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
notes_w = widgets.Textarea(
    description="Notes:",
    placeholder="Optional review notes...",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%", height="80px"),
)
log_btn = widgets.Button(description="Write validation log", button_style="primary")
signoff_out = widgets.Output()


def _write_log(_btn):
    with signoff_out:
        clear_output(wait=True)
        log_path = mqc.write_validation_log(
            layer1,
            layer2,
            config,
            log_path="docs/VALIDATION_LOG.md",
            decision=decision_w.value,
            validated_by=validated_by_w.value,
            notes=notes_w.value or "Notebook run.",
        )
        print(f"Validation log written to: {log_path}")


log_btn.on_click(_write_log)
display(Markdown("### Validation sign-off"))
display(widgets.VBox([validated_by_w, decision_w, notes_w, log_btn, signoff_out]))

## Output files

After Layer 2, outputs are written under `outputs/generated_by_script/`:

- **Report:** `qc_report_summary.md` (template-aligned summary; see **Report** tab above)
- **Excel:** `qc_report.xlsx`
- **Tables:** `tables/` including `frame_qc_mask.csv`, `unlabeled_marker_summary.csv`
- **Plots:** `plots/` including `gap_timeline.png`, `unlabeled_count_over_time.png`
- **Config copy:** `config_used.yaml`

Use the **Report** tab for a shareable QC summary close to `RAW_MOTIVE_MARKER_QC_REPORT_TEMPLATE.md`. Sections 4 (artifacts) and full BVH mapping remain for later layers.